# 🧠 Fine-tune FLAN-T5 untuk Negation Parsing (FIXED)
**Format output:** `positive: X | excluded: Y | country: Z`

### Checklist sebelum mulai:
1. Runtime → Change runtime type → **T4 GPU**
2. Jalankan semua cell dari atas ke bawah

## 📦 1. Install Dependencies

In [1]:
!pip install -q transformers datasets scikit-learn accelerate sentencepiece --upgrade

## 📁 2. Upload Dataset

In [2]:
from google.colab import files

print('Upload file synthetic_dataset.json:')
uploaded = files.upload()
dataset_filename = list(uploaded.keys())[0]
print(f'\n✅ File terupload: {dataset_filename}')

Upload file synthetic_dataset.json:


Saving synthetic_dataset.json to synthetic_dataset (2).json

✅ File terupload: synthetic_dataset (2).json


## ⚙️ 3. Konfigurasi

In [3]:
MODEL_NAME        = 'google/flan-t5-small'
DATASET_PATH      = dataset_filename
OUTPUT_DIR        = './negation_model_t5'

MAX_INPUT_LENGTH  = 128
MAX_OUTPUT_LENGTH = 128  # Naikkan dari 64 — provision name bisa panjang

EPOCHS            = 25
BATCH_SIZE        = 16
LEARNING_RATE     = 2e-4
TEST_SIZE         = 0.2
EARLY_STOPPING    = 10

print(f'  Model         : {MODEL_NAME}')
print(f'  Epochs        : {EPOCHS}')
print(f'  Batch size    : {BATCH_SIZE}')
print(f'  Learning rate : {LEARNING_RATE}')

  Model         : google/flan-t5-small
  Epochs        : 25
  Batch size    : 16
  Learning rate : 0.0002


## 🔧 4. Imports & GPU Check

In [4]:
import json
import torch
import numpy as np
from torch.utils.data import Dataset
from transformers import (
    T5ForConditionalGeneration,
    T5TokenizerFast,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from sklearn.model_selection import train_test_split

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  GPU tidak aktif! Pergi ke Runtime → Change runtime type → T4')

✅ GPU: Tesla T4


## 📊 5. Load Dataset

In [5]:
with open(DATASET_PATH, 'r') as f:
    data = json.load(f)

print(f'Total samples : {len(data)}')
print(f'Keys          : {list(data[0].keys())}')
print()
print('=== Preview 3 sample ===')
for i, item in enumerate(data[:3]):
    print(f'[{i+1}] INPUT  : {item["input"]}')
    print(f'     OUTPUT : {item["output"]}')
    print()

Total samples : 397
Keys          : ['input', 'output', 'meta']

=== Preview 3 sample ===
[1] INPUT  : Parse the trade agreement search query. Extract the positive topic and any excluded terms. Query: I need Trade in Goods but please leave out Trade in Services
     OUTPUT : positive: Trade in Goods | excluded: Trade in Services | country: none

[2] INPUT  : Parse the trade agreement search query. Extract the positive topic and any excluded terms. Query: Trade in Goods — Anti-Dumping not including Trade in Services
     OUTPUT : positive: Trade in Goods | excluded: Trade in Services | country: none

[3] INPUT  : Parse the trade agreement search query. Extract the positive topic and any excluded terms. Query: give me Digital Trade / Electronic Commerce — General Issues Under Digital Trade and ignore Quantitative Restrictions on Exports and Imports
     OUTPUT : positive: Digital Trade / Electronic Commerce | excluded: Quantitative Restrictions on Exports and Imports | country: none



## 🤖 6. Load Tokenizer & Model

In [6]:
print(f'Loading {MODEL_NAME}...')
tokenizer = T5TokenizerFast.from_pretrained(MODEL_NAME)
model     = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'✅ Model loaded — {total_params:.0f}M parameters')
print(f'   Pad token id : {tokenizer.pad_token_id}')

Loading google/flan-t5-small...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ Model loaded — 77M parameters
   Pad token id : 0


## 🧩 7. Dataset Class & Helpers
> **FIX**: Field `meta` dihapus dari `__getitem__` — ini penyebab `training_loss=0.0` dan `val_loss=NaN` sebelumnya

In [16]:
class NegationSeq2SeqDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data      = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        input_enc = self.tokenizer(
            item['input'],
            max_length=MAX_INPUT_LENGTH,
            truncation=True,
            padding='max_length',
            return_tensors='pt',
        )

        target_enc = self.tokenizer(
            item['output'],
            max_length=MAX_OUTPUT_LENGTH,
            truncation=True,
            padding='max_length',
            return_tensors='pt',
        )

        labels = target_enc['input_ids'].squeeze().clone()
        # Mask padding supaya tidak dihitung dalam loss
        labels[labels == self.tokenizer.pad_token_id] = -100

        # ✅ HANYA return 3 field ini — jangan tambahkan 'meta' atau field lain
        # DataCollatorForSeq2Seq akan error/silent-fail jika ada field tidak dikenal
        return {
            'input_ids':      input_enc['input_ids'].squeeze(),
            'attention_mask': input_enc['attention_mask'].squeeze(),
            'labels':         labels,
        }


def parse_output(text: str) -> dict:
    """Parse pipe-delimited output ke dict. Salin fungsi ini ke search_engine.py."""
    result = {'positive_query': '', 'excluded_terms': [], 'iso_code': None}
    try:
        parts = {}
        for segment in text.split('|'):
            if ':' not in segment:
                continue
            key, _, value = segment.partition(':')
            parts[key.strip()] = value.strip()

        result['positive_query'] = parts.get('positive', '').strip()

        excluded_raw = parts.get('excluded', 'none').strip()
        result['excluded_terms'] = (
            [] if excluded_raw.lower() == 'none'
            else [e.strip() for e in excluded_raw.split(',') if e.strip()]
        )

        iso_raw = parts.get('country', 'none').strip()
        result['iso_code'] = None if iso_raw.lower() == 'none' else iso_raw

    except Exception:
        pass
    return result


def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if hasattr(preds, 'dtype') and np.issubdtype(preds.dtype, np.floating):
        preds = np.argmax(preds, axis=-1)

    preds = np.where(
        (preds >= 0) & (preds < tokenizer.vocab_size),
        preds,
        tokenizer.pad_token_id
    ).astype(np.int32)

    decoded_preds  = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels         = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    exact_matches = 0
    field_matches = {'positive_query': 0, 'excluded_terms': 0, 'iso_code': 0}

    for pred, label in zip(decoded_preds, decoded_labels):
        pred, label = pred.strip(), label.strip()

        if pred == label:
            exact_matches += 1

        p = parse_output(pred)
        l = parse_output(label)

        if p['positive_query'].lower() == l['positive_query'].lower():
            field_matches['positive_query'] += 1
        if sorted(p['excluded_terms']) == sorted(l['excluded_terms']):
            field_matches['excluded_terms'] += 1
        if p['iso_code'] == l['iso_code']:
            field_matches['iso_code'] += 1

    n = len(decoded_preds)
    return {
        'exact_match':        exact_matches / n,
        'positive_query_acc': field_matches['positive_query'] / n,
        'excluded_terms_acc': field_matches['excluded_terms'] / n,
        'iso_code_acc':       field_matches['iso_code'] / n,
    }

print('✅ Dataset class & helpers siap')

✅ Dataset class & helpers siap


## 🔍 8. Sanity Check — Wajib Dijalankan Sebelum Training
> Pastikan tidak ada `loss=0` atau `labels all masked` sebelum training

In [17]:
# Buat dummy dataset untuk cek
dummy_dataset = NegationSeq2SeqDataset(data[:5], tokenizer)
sample = dummy_dataset[0]

labels        = sample['labels']
non_masked    = (labels != -100).sum().item()
total_tokens  = len(labels)

print('=== SANITY CHECK ===')
print(f'Total label tokens : {total_tokens}')
print(f'Non-masked tokens  : {non_masked}')
print(f'Masked (-100)      : {total_tokens - non_masked}')
print()

if non_masked == 0:
    print('❌ GAGAL — Semua label ter-mask! Training akan loss=0 dan val_loss=NaN')
    print('   Pastikan tidak ada field selain input_ids, attention_mask, labels')
elif non_masked < 5:
    print('⚠️  WARNING — Terlalu sedikit label yang aktif, cek output tokenization')
else:
    print(f'✅ OK — {non_masked} token aktif, training seharusnya berjalan normal')

# Decode sample untuk verifikasi
valid_labels = labels[labels != -100]
decoded = tokenizer.decode(valid_labels, skip_special_tokens=True)
print(f'\nDecoded label sample: "{decoded}"')
print(f'Expected output     : "{data[0]["output"]}"')

=== SANITY CHECK ===
Total label tokens : 128
Non-masked tokens  : 17
Masked (-100)      : 111

✅ OK — 17 token aktif, training seharusnya berjalan normal

Decoded label sample: "positive: Trade in Goods | excluded: Trade in Services | country: none"
Expected output     : "positive: Trade in Goods | excluded: Trade in Services | country: none"


## ✂️ 9. Split Dataset

In [18]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=42)

train_dataset = NegationSeq2SeqDataset(train_data, tokenizer)
test_dataset  = NegationSeq2SeqDataset(test_data,  tokenizer)

print(f'Train : {len(train_dataset)} samples')
print(f'Test  : {len(test_dataset)} samples')

Train : 317 samples
Test  : 80 samples


In [19]:
import torch
# == DEBUG: Cek apakah loss bisa dihitung manual ==
from torch.nn import CrossEntropyLoss

model.train()
model.to(device) # Move the model to the GPU
sample_batch = {
    'input_ids':      train_dataset[0]['input_ids'].unsqueeze(0).to(device),
    'attention_mask': train_dataset[0]['attention_mask'].unsqueeze(0).to(device),
    'labels':         train_dataset[0]['labels'].unsqueeze(0).to(device),
}

with torch.no_grad():
    outputs = model(**sample_batch)

print(f'Loss dari model  : {outputs.loss.item():.4f}')
print(f'Labels non-masked: {(sample_batch["labels"] != -100).sum().item()}')
print()

if outputs.loss.item() == 0.0:
    print('❌ Loss = 0 — runtime belum di-restart, class lama masih aktif')
    print('   → Runtime → Restart session → Run All dari awal')
elif outputs.loss.item() != outputs.loss.item():  # NaN check
    print('❌ Loss = NaN — kemungkinan fp16 overflow')
    print('   → Set fp16=False di training_args')
else:
    print(f'✅ Loss normal ({outputs.loss.item():.4f}) — siap training')

Loss dari model  : 0.0883
Labels non-masked: 20

✅ Loss normal (0.0883) — siap training


## 🏋️ 10. Training

In [20]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=MAX_OUTPUT_LENGTH,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='excluded_terms_acc',
    greater_is_better=True,
    logging_steps=10,
    report_to=['none'],
    fp16=False,
    bf16=torch.cuda.is_available(),
    warmup_steps=50,
    max_grad_norm=1.0,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING)],
)

print('🚀 Mulai training...')
print('   Training loss seharusnya mulai dari ~1.5–2.0 lalu turun')
print('   Jika masih 0.0 dari awal, ada masalah lain — stop dan report')
trainer.train()

🚀 Mulai training...
   Training loss seharusnya mulai dari ~1.5–2.0 lalu turun
   Jika masih 0.0 dari awal, ada masalah lain — stop dan report


Epoch,Training Loss,Validation Loss,Exact Match,Positive Query Acc,Excluded Terms Acc,Iso Code Acc
1,0.175507,0.165664,0.550000,0.687500,0.750000,0.975000
2,0.251468,0.107457,0.687500,0.775000,0.875000,0.975000
3,0.149699,0.142149,0.687500,0.775000,0.900000,0.975000
4,0.112801,0.124108,0.712500,0.775000,0.887500,0.975000
5,0.058344,0.141253,0.750000,0.800000,0.900000,1.000000
6,0.041942,0.121451,0.775000,0.825000,0.912500,1.000000
7,0.031901,0.133028,0.787500,0.837500,0.925000,1.000000
8,0.037924,0.133983,0.750000,0.787500,0.925000,1.000000
9,0.028519,0.143809,0.762500,0.812500,0.925000,1.000000
10,0.031297,0.123787,0.762500,0.837500,0.900000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=340, training_loss=0.06355182649019886, metrics={'train_runtime': 447.359, 'train_samples_per_second': 17.715, 'train_steps_per_second': 1.118, 'total_flos': 250440982462464.0, 'train_loss': 0.06355182649019886, 'epoch': 17.0})

## 📈 11. Evaluasi Final

In [22]:
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)

results = trainer.evaluate()

print('\n==============================')
print('       HASIL EVALUASI')
print('==============================')
print(f"  Exact Match         : {results.get('eval_exact_match', 0):.1%}")
print(f"  Positive Query Acc  : {results.get('eval_positive_query_acc', 0):.1%}")
print(f"  Excluded Terms Acc  : {results.get('eval_excluded_terms_acc', 0):.1%}")
print(f"  ISO Code Acc        : {results.get('eval_iso_code_acc', 0):.1%}")
print('==============================')


       HASIL EVALUASI
  Exact Match         : 78.8%
  Positive Query Acc  : 83.8%
  Excluded Terms Acc  : 92.5%
  ISO Code Acc        : 100.0%


## 💾 12. Simpan Model

In [23]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'✅ Model tersimpan di {OUTPUT_DIR}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model tersimpan di ./negation_model_t5


## 🧪 13. Quick Inference Test

In [24]:
TEST_QUERIES = [
    'Trade in Goods excluding Agriculture',
    'Rules of Origin without rice',
    'ASEAN trade remedies excluding anti-dumping and subsidies',
    'Japan Rules of Origin excluding rice',
    'Trade in Services except financial services',
    'Labour provisions not related to agriculture',
    'Intellectual Property Rights',                         # tanpa negasi
    'Digital Trade not including Paperless Trade',
    'Government Procurement leaving out State-Owned Enterprise',  # pola baru
    'Vietnam Sanitary and Phytosanitary Measures aside from automotive',
]

model.eval()
print('=' * 70)
print('INFERENCE TEST')
print('=' * 70)

for query in TEST_QUERIES:
    input_text = (
        'Parse the trade agreement search query. '
        'Extract the positive topic and any excluded terms. '
        f'Query: {query}'
    )

    inputs = tokenizer(
        input_text,
        return_tensors='pt',
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_OUTPUT_LENGTH,
            num_beams=4,
            early_stopping=True,
        )

    raw    = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    parsed = parse_output(raw)

    print(f'\nQuery    : {query}')
    print(f'Raw      : {raw}')
    print(f'Positive : {parsed["positive_query"]}')
    print(f'Excluded : {parsed["excluded_terms"]}')
    print(f'Country  : {parsed["iso_code"]}')

Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFERENCE TEST


Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Query    : Trade in Goods excluding Agriculture
Raw      : positive: Trade in Goods | excluded: Agriculture | country: none
Positive : Trade in Goods
Excluded : ['Agriculture']
Country  : None


Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


Query    : Intellectual Property Rights
Raw      : positive: Intellectual Property Rights | excluded: none | country: none
Positive : Intellectual Property Rights
Excluded : []
Country  : None


Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Query    : Digital Trade not including Paperless Trade
Raw      : positive: Digital Trade | excluded: Paperless Trade | country: none
Positive : Digital Trade
Excluded : ['Paperless Trade']
Country  : None


Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Query    : Government Procurement leaving out State-Owned Enterprise
Raw      : positive: Government Procurement | excluded: State-Owned Enterprise | country: none
Positive : Government Procurement
Excluded : ['State-Owned Enterprise']
Country  : None

Query    : Vietnam Sanitary and Phytosanitary Measures aside from automotive
Raw      : positive: Vietnam Sanitary and Phytosanitary Measures | excluded: automotive | country: VNM
Positive : Vietnam Sanitary and Phytosanitary Measures
Excluded : ['automotive']
Country  : VNM


## ⬇️ 14. Download Model

In [25]:
import shutil
import os
from google.colab import files

zip_path = 'negation_model_t5'
temp_dir = '/tmp/negation_model_export'

# Copy hanya file di root, skip folder checkpoint
os.makedirs(temp_dir, exist_ok=True)
for item in os.listdir(OUTPUT_DIR):
    if item.startswith('checkpoint'):
        continue
    src = os.path.join(OUTPUT_DIR, item)
    dst = os.path.join(temp_dir, item)
    if os.path.isfile(src):
        shutil.copy2(src, dst)

shutil.make_archive(zip_path, 'zip', temp_dir)
print(f'✅ Model di-zip: {zip_path}.zip')
files.download(f'{zip_path}.zip')

✅ Model di-zip: negation_model_t5.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>